STEP 0 - Import libraries

In [2]:
import pandas as pd
from datetime import datetime
import sqlite3

STEP #1 - Connect to database

In [3]:
conn = sqlite3.connect('../data/ecommerce.db')

query = """
SELECT * 
FROM transactions_clean 
"""

df = pd.read_sql(query, conn)
display(df.head(3))
display(df.info())


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Year,Month,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,2010-12


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392692 entries, 0 to 392691
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    392692 non-null  object 
 1   StockCode    392692 non-null  object 
 2   Description  392692 non-null  object 
 3   Quantity     392692 non-null  int64  
 4   InvoiceDate  392692 non-null  object 
 5   UnitPrice    392692 non-null  float64
 6   CustomerID   392692 non-null  float64
 7   Country      392692 non-null  object 
 8   TotalPrice   392692 non-null  float64
 9   Year         392692 non-null  int64  
 10  Month        392692 non-null  int64  
 11  YearMonth    392692 non-null  object 
dtypes: float64(3), int64(3), object(6)
memory usage: 36.0+ MB


None

STEP #2 - Convert to datetime and define snapshot date

In [4]:
# Convert InvoiceDate column to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Define snapshot date
snapshot_date = df['InvoiceDate'].max()

STEP #3 - Calculate RFM

In [5]:
# Calculate recency, frequency, monetary
rfm = df.groupby('CustomerID').agg(recency = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days), 
frequency = ('InvoiceNo', 'nunique'), monetary = ('TotalPrice', 'sum')).reset_index()

# Convert CustomerID to integer type
rfm['CustomerID'] = rfm['CustomerID'].astype(int)
display(rfm)

,CustomerID,recency,frequency,monetary
0,12346,325,1,77183.60
1,12347,1,7,4310.00
2,12348,74,4,1797.24
3,12349,18,1,1757.55
4,12350,309,1,334.40
...,...,...,...,...
4333,18280,277,1,180.60
4334,18281,180,1,80.82
4335,18282,7,2,178.05
4336,18283,3,16,2045.53


STEP #4 - Score each metric 1-5 using quintiles and create score columns

In [6]:


# recency score
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels = [5,4,3,2,1], duplicates='drop')

# frequency score; can't create 5 equal bins due to duplicate frequencies, labels=False auto-adjusts
rfm['f_score'] = pd.qcut(rfm['frequency'], q=5, labels=False, duplicates='drop') + 1

# monetary score
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels = [1,2,3,4,5], duplicates='drop')

display(rfm)


,CustomerID,recency,frequency,monetary,r_score,f_score,m_score
0,12346,325,1,77183.60,1,1,5
1,12347,1,7,4310.00,5,4,5
2,12348,74,4,1797.24,2,3,4
3,12349,18,1,1757.55,4,1,4
4,12350,309,1,334.40,1,1,2
...,...,...,...,...,...,...,...
4333,18280,277,1,180.60,1,1,1
4334,18281,180,1,80.82,1,1,1
4335,18282,7,2,178.05,5,1,1
4336,18283,3,16,2045.53,5,4,4


STEP #5 - Create segment labels based on each score

In [7]:
def labels(row):
    if row['r_score'] >= 4 and row['f_score'] >= 4 and row['m_score'] >= 4:
        return 'Champion'
    elif row['r_score'] >= 3 and row['f_score'] >= 3:
        return 'Loyal Customer'
    elif row['r_score'] >= 4 and  row['f_score'] <= 2:
        return 'New Customer'
    elif row['r_score'] <= 2 and row['f_score'] >= 3:
        return 'At Risk'
    elif row['r_score'] <= 2 and row['f_score'] <= 2:
        return 'Lost'
    else:
        return 'Potential Loyalist'

rfm['segment'] = rfm.apply(labels, axis=1)
display(rfm)



,CustomerID,recency,frequency,monetary,r_score,f_score,m_score,segment
0,12346,325,1,77183.60,1,1,5,Lost
1,12347,1,7,4310.00,5,4,5,Champion
2,12348,74,4,1797.24,2,3,4,At Risk
3,12349,18,1,1757.55,4,1,4,New Customer
4,12350,309,1,334.40,1,1,2,Lost
...,...,...,...,...,...,...,...,...
4333,18280,277,1,180.60,1,1,1,Lost
4334,18281,180,1,80.82,1,1,1,Lost
4335,18282,7,2,178.05,5,1,1,New Customer
4336,18283,3,16,2045.53,5,4,4,Champion


STEP #6 - Save rfm_analysis table to SQL database

In [8]:
rfm.to_sql(name='rfm_analysis', con=conn, if_exists='replace', index=False)

4338

STEP #7 - Calculate averages of each segment and total revenue per segment; add columns to rfm dataframe

In [9]:
display(rfm['segment'].describe(), rfm['segment'].info())

segment_summary = rfm.groupby('segment').agg(num_customers = ('CustomerID', 'count'), avg_rec = ('recency', 'mean'), avg_freq = ('frequency', 'mean'), avg_mon = ('monetary', 'mean'), total_rev = ('monetary', 'sum'))

display(segment_summary)

<class 'pandas.core.series.Series'>
RangeIndex: 4338 entries, 0 to 4337
Series name: segment
Non-Null Count  Dtype 
--------------  ----- 
4338 non-null   object
dtypes: object(1)
memory usage: 34.0+ KB


count     4338
unique       6
top       Lost
freq      1505
Name: segment, dtype: object

None

,num_customers,avg_rec,avg_freq,avg_mon,total_rev
segment,,,,,
At Risk,203,130.339901,5.561576,1826.597438,370799.280
Champion,573,10.064572,15.404887,8641.831640,4951769.530
Lost,1505,200.779402,1.483721,628.289291,945575.383
Loyal Customer,726,26.928375,5.356749,2154.361062,1564066.131
New Customer,737,16.572592,1.956581,916.134532,675191.150
Potential Loyalist,594,52.057239,1.703704,639.406431,379807.420


In [10]:
display(rfm)

,CustomerID,recency,frequency,monetary,r_score,f_score,m_score,segment
0,12346,325,1,77183.60,1,1,5,Lost
1,12347,1,7,4310.00,5,4,5,Champion
2,12348,74,4,1797.24,2,3,4,At Risk
3,12349,18,1,1757.55,4,1,4,New Customer
4,12350,309,1,334.40,1,1,2,Lost
...,...,...,...,...,...,...,...,...
4333,18280,277,1,180.60,1,1,1,Lost
4334,18281,180,1,80.82,1,1,1,Lost
4335,18282,7,2,178.05,5,1,1,New Customer
4336,18283,3,16,2045.53,5,4,4,Champion


In [11]:
rfm.to_csv('../data/rfm_analysis.csv', index=False)